In [1]:
import re
import networkx as nx
import pickle
import pandas as pd
import ast
from pykeen.triples import TriplesFactory
from pykeen.hpo import hpo_pipeline_from_path
from pykeen.sampling import NegativeSampler
from pykeen.pipeline import pipeline
import torch
import numpy as np

/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
pattern_dash = r'(\w+)_HUMAN'
def gene_symbol_extractor(text, pattern:str):
        # ^ ensures start at the beginning, $ ensures end at the ')'
        match = re.search(pattern, text)
        if match:
            target = match.group(1)
            return target.upper()
        return None

In [9]:
def load_graph(filepath="../datasets/PPI_KGs/G_adni_ADKG_all.pkl"):
    with open(filepath,'rb') as f:
        ppikg = pickle.load(f)
    print(f"Load Graph with {ppikg.number_of_nodes()} nodes and {ppikg.number_of_edges()} edges.")
    return ppikg

## Get train_pos/neg edegs

In [11]:
df_adni_dual_all = pd.read_csv("../datasets/Prime_KGs/Summary_adni_dual_hybrid_all.csv", index_col=0)
df_adni_dual_all

,pos_edges_disease,neg_edges_disease,pos_edges_healthy,neg_edges_healthy,train,val,test,linked_disease_nodes,linked_healthy_nodes,label
116_S_1249,381,425,96,114,False,True,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",1
037_S_4410,310,496,66,144,False,True,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",0
006_S_4153,356,450,90,120,False,False,True,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",1
116_S_1232,411,395,125,85,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",0
128_S_0205,325,481,78,132,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",0
...,...,...,...,...,...,...,...,...,...,...
014_S_4668,405,401,110,100,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",1
130_S_0289,367,439,78,132,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",1
009_S_2381,415,391,126,84,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",1
041_S_4014,434,372,123,87,True,False,False,"['AAMP', 'AATF', 'ABAT', 'ACACA', 'ACADL', 'AC...","['p(UniProtKB:""ABCA1_MOUSE"")', 'p(UniProtKB:""A...",0


In [12]:
def get_pos_neg_edges(df):
    train_pos, train_neg = [], []
    val_pos, val_neg = [], []
    test_pos, test_neg = [], [] 

    train_label_pos, train_label_neg = [], []
    val_label_pos, val_label_neg = [], []
    test_label_pos, test_label_neg = [], [] 
    # Grouping the positive and negative lists into a dictionary mapping for clean access
    splits = {
        'train': (train_pos, train_neg),
        'val': (val_pos, val_neg),
        'test': (test_pos, test_neg)
    }
    label_splits = {
        'train': (train_label_pos, train_label_neg),
        'val': (val_label_pos, val_label_neg),
        'test': (test_label_pos, test_label_neg)
    }

    for idx, row in df.iterrows():
        patient_node = idx
        
        # 1. Determine which split this row belongs to
        active_split = None
        for split_name in splits:
            if row[split_name]:
                active_split = split_name
                break
                
        if active_split is None:
            print(f"KeyError: Row {idx} does not belong to train, val, or test.")
            continue
            
        # 2. Extract lists
        disease_dst = ast.literal_eval(row['linked_disease_nodes'])
        healthy_dst = ast.literal_eval(row['linked_healthy_nodes'])
        
        # 3. Dynamically assign positive and negative target arrays based on the split
        pos_list, neg_list = splits[active_split]
        pos_label_list, neg_label_list = label_splits[active_split]
        
        # 4. Route the edges cleanly based on the label
        if row['label'] == 1:
            pos_list.extend((patient_node, 'express', dst) for dst in disease_dst)
            neg_list.extend((patient_node, 'express', dst) for dst in healthy_dst)
           
            pos_label_list.extend((patient_node, 'reg_disease', dst) for dst in disease_dst)
            neg_label_list.extend((patient_node, 'reg_disease', dst) for dst in healthy_dst)
        else:
            neg_list.extend((patient_node, 'express', dst) for dst in disease_dst)
            pos_list.extend((patient_node, 'express', dst) for dst in healthy_dst)

            neg_label_list.extend((patient_node, 'reg_healthy', dst) for dst in disease_dst)
            pos_label_list.extend((patient_node, 'reg_healthy', dst) for dst in healthy_dst)
    
    return splits, label_splits

+ (disease_sample_i, reg_disease, disease_protein)  → POSITIVE ✅
+ (disease_sample_i, reg_disease, healthy_protein)  → NEGATIVE ❌  wrong protein ----> most informative
+ (disease_sample_i, reg_healthy, disease_protein)  → NEGATIVE ❌  wrong relation
+ (disease_sample_i, reg_healthy, healthy_protein)  → NEGATIVE ❌  wrong both

In [13]:
adni_dual_all_splits, adni_dual_all_label_splits = get_pos_neg_edges(df=df_adni_dual_all)
adni_dual_all_splits

{'train': ([('116_S_1232', 'express', 'p(UniProtKB:"ABCA1_MOUSE")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ABCA7_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ABCG1_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ABCG4_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ABL1_MOUSE")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ACTN2_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"ADAM9_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"AKT1_MOUSE")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APBA1_MOUSE")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APBA2_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APBB1_RAT")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APH1A_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APH1B_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APLP1_MOUSE")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APLP2_HUMAN")'),
   ('116_S_1232', 'express', 'p(UniProtKB:"APOA1_MOUSE")'),
   ('116_S_1232', 'express', 'p(Uni

In [14]:
adni_dual_all_label_splits

{'train': ([('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ABCA1_MOUSE")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ABCA7_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ABCG1_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ABCG4_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ABL1_MOUSE")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ACTN2_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"ADAM9_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"AKT1_MOUSE")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APBA1_MOUSE")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APBA2_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APBB1_RAT")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APH1A_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APH1B_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APLP1_MOUSE")'),
   ('116_S_1232', 'reg_healthy', 'p(UniProtKB:"APLP2_HUMAN")'),
   ('116_S_1232', 'reg_healthy', 'p

In [15]:
G = load_graph("../datasets/Prime_KGs/G_adni_dual_hybrid_all.pkl")

kg_triples=[]
for u, v, rel, data in G.edges(data=True, keys=True):
    
    # 1. Fast path: check if the edge key 'rel' is a valid string
    if isinstance(rel, str):
        relation = rel
    else:
        # 2. Fallback chain using dict.get() defaults
        relation = data.get('type') or data.get('relation') or data.get('rel')

    # 3. Guard against NoneType before checking 'rev'
    if relation and 'rev' not in relation:
        if u not in list(df_adni_dual_all.index):
            kg_triples.append((u, relation, v))
        else:
            if relation == 'similar':
                kg_triples.append((u, relation, v))
            

Load Graph with 9525 nodes and 962247 edges.


In [58]:
agnostic_triples = []

for k,v in adni_dual_all_splits.items():
    agnostic_triples.extend(v[0])
    agnostic_triples.extend(v[1])
    
train_label_triples = adni_dual_all_label_splits['train'][0]

In [60]:
agnostic_triples
train_label_triples

[('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ABCA1_MOUSE")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ABCA7_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ABCG1_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ABCG4_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ABL1_MOUSE")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ACTN2_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"ADAM9_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"AKT1_MOUSE")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APBA1_MOUSE")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APBA2_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APBB1_RAT")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APH1A_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APH1B_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APLP1_MOUSE")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APLP2_HUMAN")'),
 ('037_S_4410', 'reg_healthy', 'p(UniProtKB:"APOA1_MOUSE")'),
 ('037_S_441

## Train PyKEEN

In [65]:
train_triples = np.concatenate([
    kg_triples,
    agnostic_triples,       # includes val/test sample nodes → they get embeddings
    train_label_triples,    # only train samples' positive label edges
], axis=0)

tf_train = TriplesFactory.from_labeled_triples(
    triples=train_triples,
    create_inverse_triples=True,
)

# --- Run KGE ---
ratios = [0.8, 0.1, 0.1]
train, test, val = tf_train.split(ratios, random_state=42)

config_file = "../PyKeen/configs/RotatE_config_hpo.json"  
print(f"\n--- Running HPO---\n")
hpo_result = hpo_pipeline_from_path(
    config_file, 
    training = train, 
    testing=test, 
    validation=val
    )
        

[I 2026-05-26 17:23:28,099] A new study created in memory with name: no-name-61c7c894-a47f-4d30-a574-b40329fbd887
/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/optuna/distributions.py:671: UserWarning: The distribution is specified by [1, 20] and step=5.0, but the range is not divisible by `step`. It will be replaced with [1, 16.0].
  optuna_warn(
/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/optuna/distributions.py:671: UserWarning: The distribution is specified by [0.1, 1.0] and step=0.5, but the range is not divisible by `step`. It will be replaced with [0.1, 0.6].
  optuna_warn(
/home/xyu/.conda/envs/firegnn/lib/python3.11/site-packages/optuna/distributions.py:684: UserWarning: The distribution is specified by [1, 50] and step=10, but the range is not divisible by `step`. It will be replaced with [1, 41].
  optuna_warn(
No random seed is specified. Setting to 1682555331.



--- Running HPO---



INFO:pykeen.triples.triples_factory:Creating inverse triples.
Training epochs on cuda:0:  24%|██▍       | 24/100 [19:42<1:03:50, 50.40s/epoch, loss=5.54, prev_loss=5.54]INFO:pykeen.evaluation.evaluator:Evaluation took 87.99s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 25: 0.013673112311080274. Saved model weights to /home/xyu/.data/pykeen/checkpoints/best-model-weights-53a5b1dc-e71c-46f6-9001-183caa13c3f6.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 25.
Training epochs on cuda:0: 100%|██████████| 100/100 [1:26:45<00:00, 52.05s/epoch, loss=5.54, prev_loss=5.54]
Evaluating on cuda:0: 100%|██████████| 66.3k/66.3k [02:07<00:00, 518triple/s]  
INFO:pykeen.evaluation.evaluator:Evaluation took 128.92s seconds
[I 2026-05-26 18:52:24,477] Trial 0 finished with value: 0.012029020483272496 and parameters: {'model.embedding_dim': 256, 'loss.margin': 11.0, 'loss.adversarial_temperature': 0.1, 'optimizer.lr': 0.45383915072637565, 'neg

In [77]:
hpo_result.save_to_directory("../PyKeen/results/dual_kg")

In [80]:
import pandas as pd
from pykeen.predict import predict_target, predict_triples
from pykeen.pipeline import pipeline_from_path

# 2. RETRAIN THE MODEL WITH BEST PARAMETERS
print("\n--- Retraining Best Model Configuration ---")

pipeline_config = "../PyKeen/results/dual_kg/best_pipeline/pipeline_config.json"
best_pipeline_result = pipeline_from_path(
    pipeline_config,
    training=train,
    testing=test,
    validation=val
)

INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding(
  (regularizer): NoRegularizer()
)
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.triples.triples_factory:Creating inverse triples.



--- Retraining Best Model Configuration ---


Training epochs on cuda:0: 100%|██████████| 10/10 [08:05<00:00, 48.56s/epoch, loss=5.56, prev_loss=5.54]
Evaluating on cuda:0: 100%|██████████| 66.3k/66.3k [01:21<00:00, 814triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 82.51s seconds


In [81]:
best_model = best_pipeline_result.model
best_pipeline_result.save_to_directory("../PyKeen/results/dual_kg")

INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=9511, num_relations=86, create_inverse_triples=True, num_triples=530381) to file:///home/xyu/thesis/HybridKG/PyKeen/results/dual_kg/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /home/xyu/thesis/HybridKG/PyKeen/results/dual_kg


In [90]:
import json
from pathlib import Path
from pykeen.pipeline import pipeline_from_config


pipeline_config = json.loads(
    Path("../PyKeen/results/dual_kg/best_pipeline/pipeline_config.json").read_text()
    )
pipeline_config['pipeline']['training']=train
pipeline_config['pipeline']['testing']=test
pipeline_config['pipeline']['validation']=val

best_pipeline_result = pipeline_from_config(pipeline_config)

INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding(
  (regularizer): NoRegularizer()
)
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.triples.triples_factory:Creating inverse triples.
Training epochs on cuda:0:   0%|          | 0/10 [00:05<?, ?epoch/s]


KeyboardInterrupt: 

In [84]:
# 3. LINK PREDICTION

print("\n--- Running Link Prediction Workflows ---")
candidate_triples = np.concatenate([adni_dual_all_label_splits['val'][0],
                                    adni_dual_all_label_splits['val'][1],
                                    adni_dual_all_label_splits['test'][0],
                                    adni_dual_all_label_splits['test'][1],
                                    ],axis=0)

# Scores the specific combinations
scored_pack = predict_triples(
    model=best_model, 
    triples=candidate_triples, 
    triples_factory=train
)
# Convert the scored pack to a readable Pandas DataFrame
predictions_df = scored_pack.process(factory=train).df
print("\nScores for specific candidate triples:")
predictions_df


--- Running Link Prediction Workflows ---

Scores for specific candidate triples:


,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score
0,0,002_S_0413,38,reg_healthy,456,AAMP,-19.100306
1,0,002_S_0413,38,reg_healthy,458,AATF,-23.603207
2,0,002_S_0413,38,reg_healthy,459,ABAT,-16.800436
3,0,002_S_0413,38,reg_healthy,482,ACACA,-12.660259
4,0,002_S_0413,38,reg_healthy,485,ACADL,-18.646162
...,...,...,...,...,...,...,...
139187,451,941_S_4292,38,reg_healthy,8527,"p(UniProtKB:""VAMP3_HUMAN"")",-11.282839
139188,451,941_S_4292,38,reg_healthy,8530,"p(UniProtKB:""VAMP8_HUMAN"")",-16.731258
139189,451,941_S_4292,38,reg_healthy,8537,"p(UniProtKB:""VIP_HUMAN"")",-18.566980
139190,451,941_S_4292,38,reg_healthy,8541,"p(UniProtKB:""VLDLR_MOUSE"")",-12.849744


In [86]:
sample_label_map = df_adni_dual_all['label'].to_dict()
predictions_df['true_labels'] = predictions_df['head_label'].map(sample_label_map)
predictions_df

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
0,0,002_S_0413,38,reg_healthy,456,AAMP,-19.100306,0
1,0,002_S_0413,38,reg_healthy,458,AATF,-23.603207,0
2,0,002_S_0413,38,reg_healthy,459,ABAT,-16.800436,0
3,0,002_S_0413,38,reg_healthy,482,ACACA,-12.660259,0
4,0,002_S_0413,38,reg_healthy,485,ACADL,-18.646162,0
...,...,...,...,...,...,...,...,...
139187,451,941_S_4292,38,reg_healthy,8527,"p(UniProtKB:""VAMP3_HUMAN"")",-11.282839,0
139188,451,941_S_4292,38,reg_healthy,8530,"p(UniProtKB:""VAMP8_HUMAN"")",-16.731258,0
139189,451,941_S_4292,38,reg_healthy,8537,"p(UniProtKB:""VIP_HUMAN"")",-18.566980,0
139190,451,941_S_4292,38,reg_healthy,8541,"p(UniProtKB:""VLDLR_MOUSE"")",-12.849744,0


In [88]:
predictions_df[predictions_df['relation_label']=="reg_healthy"]

,head_id,head_label,relation_id,relation_label,tail_id,tail_label,score,true_labels
0,0,002_S_0413,38,reg_healthy,456,AAMP,-19.100306,0
1,0,002_S_0413,38,reg_healthy,458,AATF,-23.603207,0
2,0,002_S_0413,38,reg_healthy,459,ABAT,-16.800436,0
3,0,002_S_0413,38,reg_healthy,482,ACACA,-12.660259,0
4,0,002_S_0413,38,reg_healthy,485,ACADL,-18.646162,0
...,...,...,...,...,...,...,...,...
139187,451,941_S_4292,38,reg_healthy,8527,"p(UniProtKB:""VAMP3_HUMAN"")",-11.282839,0
139188,451,941_S_4292,38,reg_healthy,8530,"p(UniProtKB:""VAMP8_HUMAN"")",-16.731258,0
139189,451,941_S_4292,38,reg_healthy,8537,"p(UniProtKB:""VIP_HUMAN"")",-18.566980,0
139190,451,941_S_4292,38,reg_healthy,8541,"p(UniProtKB:""VLDLR_MOUSE"")",-12.849744,0


In [91]:
predictions_df.to_csv("../PyKeen/results/dual_kg/link_predictions.csv")